# 2D object tracking workshop

Download the zipped data from the blackboard page. This data is taken from the cell tracking challenge: (https://celltrackingchallenge.net). 
This is a widely used source to benchmark the performance of tracking algorithms (both classical and using AI) - see e.g. (https://www.nature.com/articles/s41592-023-01879-y).
We are going to use some of the techniques we have covered across recent classes to interpret the behaviour of a HL60 cell line, including size, growth rate and average separation. Each image is taken 30 minutes apart, and the pixel size is 0.125 micrometres.

First, unzip the dataset to a 'data' folder in the same location as this notebook. You can view the data as a video by opening imagej (in apps anywhere) and using file/import image sequence, which can then import the entire folder. You may need to adjust the brightness of the image by using image/adjust brightness/contrast.
Next, run the imports below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import scipy as sp
import trackpy as tp
import pandas as pd
import pims

from pims import ImageSequence

If you receive an error in the following cell, check the directory name matches the location on your system. If the below cell throws an error, it could be because imagecodecs is not installed - install via pip/condaforge or equivalent, then restart the kernel. You may also see a non-fatal warning related to skimage - this is a warning that some of the syntax will become out of date in the library.

In [ ]:
frames = ImageSequence('data/*.tif')

We can view the metadata for the dataset simply by typing 'frames'. Check this matches your expectations (is the location correct, the number of frames, the frame shape etc.?)

In [ ]:
frames

We can then view a specific frame (x) either by typing frames[x] or we can plot with a bit more control (and the ability to save our images) using pyplot, as well as the option of using different colourmaps.

In [ ]:
frames[2]

In [ ]:
plt.imshow(sp.ndimage.uniform_filter(frames[0],size=5),cmap='prism')

plt.savefig('test1.jpg')
plt.show()

We can then attempt to locate features using tp.locate. Initially, the most basic input it takes is the approximate size of the features.
We can then annotate that frame to see how well we have tracked the features. Try changing the value for the object size (which should be an odd number for fitting accuracy) and see how it changes the reliability of the tracking. Then, change the value of x to check that the parameters are suitable for the entire dataset.

In [ ]:
x=0
f = tp.locate(frames[x], 51,invert=False)
annotatedframe=tp.annotate(f, frames[x])


Our output from trackpy includes multiple properties for each object, and we can filter by each of these to either improve the quality of the tracking or to interpret the data. We can view all these properties with f.head(). For example, we could plot the range of "masses" (this is the total intensity of the object) to see if there are any especially bright or dim objects. Very dim objects could be false positives, so we can filter these out. We do this by adding minmass=y whatever value you think the minimum realistic intensity is to the tp.locate function. Similarly, we can add a blur function (similar to sp.ndimage.uniformfilter) with smoothing_size=z - see the API reference for more information http://soft-matter.github.io/trackpy/v0.7/generated/trackpy.locate.html

In [ ]:
f.head()

In [ ]:
	
fig, ax = plt.subplots()
ax.hist(f['mass'], bins=20)

# Optionally, label the axes.
ax.set(xlabel='mass', ylabel='count')
plt.show()

When we are happy with the parameters we have chosen for tracking (above) then we can proceed to batch process all the data in one go. Update the object size, minmass and any other parameters you may have selected in the below cell and run tp.batch.

In [ ]:
f = tp.batch(frames[:], 51, invert=False, minmass=0)

This produces one large dataframe containing data from all of the individual images. We can select the tracked objects in frame 'x' in the following cell:

In [ ]:
x=10

fx= f[f['frame']==x]

In [ ]:
fx.head()

# Tasks

1) Find reliable values for the parameters (size, minmass etc.) in trackpy to accurately track cell positions for the the first frame. Check that these parameters are still appropriate for frames later on in the dataset (e.g., frames 25,50,75 etc). When you have optimised, batch process the entire dataset 

2) By selecting the dataframes one by one (or by other means) produce a plot of the number of cells as a function of frame number.

3. Produce a second plot of number of cells versus frame number, for four values of smoothing_size: your object size (that you input to trackpy), double your object size, half your object size and a quarter of your object size. You may need to set a new min_mass for each. Which values for smoothing_size yield give the best recall of the image?

4. From the dataset corresponding to your best tracking conditions, determine the growth rate of the cells, noting that each frame was taken 30 minutes after the previous.

5. From the dataset corresponding to your best tracking conditions, use the built in trackpy function (http://soft-matter.github.io/trackpy/v0.7/generated/trackpy.static.pair_correlation_2d.html) to determine a time-averaged radial distribution function for all 109 frames. The x axis should be in microns, not pixels. What does the RDF tell us about the interactions between the cells?